In [1]:
from fuseqa import SRE
sre = SRE()

INFO:fuseqa.sre:Loading tokenizer...
INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/gemma-2-2b-it/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/gemma-2-2b-it/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google/gemma-2-2b-it/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google/gemma-2-2b-it/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:fuseqa.sre:Loading BASE model...
INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/gemma-2-2b-it/resolve/main/config.json "HTTP/1.1 200 OK"
`torch_dtype` is deprecated! Use `dtype` instead!
INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/gemma-2-2b-it/resolve/main/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Gemma2ForSequenceClassification LOAD REPORT from: google/gemma-2-2b-it
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
INFO:fuseqa.sre:Loading LoRA model...
INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/gemma-2-2b-it/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/gemma-2-2b-it/resolve/main/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Gemma2ForSequenceClassification LOAD REPORT from: google/gemma-2-2b-it
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/MinaGabriel/gemma-2-2b-it-SRE-LoRA/resolve/main/adapter_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/MinaGabriel/gemma-2-2b-it-SRE-LoRA/0b79210a53f110f99e70dc8209a88b4d3292250e/adapter_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/MinaGabriel/gemma-2-2b-it-SRE-LoRA/resolve/main/adapter_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/MinaGabriel/gemma-2-2b-it-SRE-LoRA/0b79210a53f110f99e70dc8209a88b4d3292250e/adapter_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Reques

In [3]:
# =========================
# EXAMPLE
# =========================

example = {
    "question": "Which company created the Java programming language?",
    "retrieved_docs": [
        # Correct but indirect
        "Java was originally developed by James Gosling at Sun Microsystems in the 1990s.",

        # Misleading (acquisition confusion)
        "Oracle Corporation acquired Sun Microsystems in 2010 and now maintains Java.",

        # Strong lexical trap
        "Microsoft develops programming languages such as C# and TypeScript.",

        # Another distractor
        "Google develops Java-related tools and uses Java in Android development.",

        # Irrelevant but domain-related
        "Python is a widely used high-level programming language.",

        # Partial supporting signal
        "Sun Microsystems was a major technology company known for Java."
    ]
}

q = example["question"]

# =========================
# CANDIDATE SENTENCES
# =========================

sentences = [
    # Correct answer (requires reasoning: Gosling → Sun)
    "Sun Microsystems created Java.",

    # Very strong distractor (ownership confusion)
    "Oracle created Java.",

    # Lexical trap (programming + big company)
    "Microsoft created Java.",

    # Domain confusion
    "Google created Java.",

    # Supporting but not direct answer
    "Java was developed by James Gosling.",

    # Irrelevant but plausible
    "Python is a popular programming language."
]

# =========================
# RUN
# =========================

full_context = "\n".join(example["retrieved_docs"])

print("\nComparing BASE vs LoRA...\n")

results = sre.compare_base_vs_lora(q, full_context, sentences)

sre.print_table(results)


Comparing BASE vs LoRA...

+------+----------+----------+----------+-------------------------------------------+
| Rank | Base Yes | LoRA Yes | Δ Impact |                  Sentence                 |
+------+----------+----------+----------+-------------------------------------------+
|  1   |  0.9507  |  0.9546  | +0.0039  |       Sun Microsystems created Java.      |
|  2   |  0.9478  |  0.9121  | -0.0356  |    Java was developed by James Gosling.   |
|  3   |  0.9033  |  0.6943  | -0.2090  |            Oracle created Java.           |
|  4   |  0.8584  |  0.3052  | -0.5532  |          Microsoft created Java.          |
|  5   |  0.8608  |  0.1409  | -0.7197  |            Google created Java.           |
|  6   |  0.9360  |  0.0015  | -0.9346  | Python is a popular programming language. |
+------+----------+----------+----------+-------------------------------------------+
